In [ ]:
!pip install faker pandas numpy
!pip install datetime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 16.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [ ]:
fake=Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)
#configruation
NUM_USERS=2500
NUM_PRODUCT=150
NUM_ORDERS=11000
print("Generating realistic e-commerce data")

Generating realistic e-commerce data


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# ---- CONFIGURATION ----
NUM_USERS = 2500
NUM_PRODUCTS = 150
NUM_ORDERS = 11000

print("Generating realistic e-commerce data...")

# 1. USERS TABLE
users = []
start_date = datetime(2025, 1, 1)
for i in range(1, NUM_USERS + 1):
    created_at = fake.date_between_dates(date_start=start_date, date_end=datetime(2026, 6, 1))
    users.append({
        'user_id': i,
        'name': fake.name(),
        'email': fake.unique.email(),
        'country': random.choices(['India', 'USA', 'UK', 'Canada', 'Germany'], weights=[60, 15, 10, 10, 5])[0],
        'created_at': created_at
    })
df_users = pd.DataFrame(users)

# 2. PRODUCTS TABLE
categories = {
    'Electronics': ['Smartphone', 'Laptop', 'Wireless Earbuds', 'Smartwatch', 'Power Bank'],
    'Apparel': ['T-Shirt', 'Jeans', 'Hoodie', 'Sneakers', 'Socks'],
    'Home Decor': ['Desk Lamp', 'Wall Art', 'Throw Pillow', 'Scented Candle', 'Indoor Plant']
}
products = []
p_id = 1
for cat, subcats in categories.items():
    for subcat in subcats:
        for _ in range(10):
            price = round(random.uniform(10.0, 120.0), 2) if cat != 'Electronics' else round(random.uniform(50.0, 999.0), 2)
            products.append({
                'product_id': p_id,
                'product_name': f"{fake.color_name()} {subcat}",
                'category': cat,
                'price': price
            })
            p_id += 1
df_products = pd.DataFrame(products)

# 3. ORDERS & ORDER ITEMS
orders = []
order_items = []
o_id = 1
oi_id = 1

user_ids = df_users['user_id'].tolist()
product_ids = df_products['product_id'].tolist()
product_prices = df_products.set_index('product_id')['price'].to_dict()

for _ in range(NUM_ORDERS):
    user_id = random.choices(user_ids, k=1)[0]
    user_reg_date = pd.to_datetime(df_users.loc[df_users['user_id'] == user_id, 'created_at'].values[0])

    order_date = fake.date_between_dates(date_start=user_reg_date, date_end=datetime(2026, 6, 15))
    status = random.choices(['Completed', 'Cancelled', 'Returned'], weights=[85, 10, 5])[0]

    orders.append({
        'order_id': o_id,
        'user_id': user_id,
        'order_date': order_date,
        'status': status
    })

    num_items = random.choices([1, 2, 3, 4], weights=[60, 25, 10, 5])[0]
    chosen_products = random.sample(product_ids, num_items)

    for prod_id in chosen_products:
        order_items.append({
            'order_item_id': oi_id,
            'order_id': o_id,
            'product_id': prod_id,
            'quantity': random.choices([1, 2, 3], weights=[85, 12, 3])[0],
            'price_per_unit': product_prices[prod_id]
        })
        oi_id += 1
    o_id += 1

df_orders = pd.DataFrame(orders)
df_order_items = pd.DataFrame(order_items)

# ---- EXPORT TO SQL ----
def to_sql_inserts(df, table_name):
    sql_strs = []
    for _, row in df.iterrows():
        values = []
        for val in row:
            if pd.isna(val): values.append("NULL")
            elif isinstance(val, (int, float)): values.append(str(val))
            else: values.append(f"'{str(val).replace(chr(39), chr(39)+chr(39))}'")
        sql_strs.append(f"INSERT INTO {table_name} VALUES ({', '.join(values)});")
    return sql_strs

with open('ecommerce_dataset.sql', 'w', encoding='utf-8') as f:
    f.write("CREATE DATABASE IF NOT EXISTS ecommerce_analytics;\nUSE ecommerce_analytics;\n\n")
    f.write("""
CREATE TABLE users (user_id INT PRIMARY KEY, name VARCHAR(100), email VARCHAR(100), country VARCHAR(50), created_at DATE);
CREATE TABLE products (product_id INT PRIMARY KEY, product_name VARCHAR(150), category VARCHAR(100), price DECIMAL(10,2));
CREATE TABLE orders (order_id INT PRIMARY KEY, user_id INT, order_date DATE, status VARCHAR(50));
CREATE TABLE order_items (order_item_id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, price_per_unit DECIMAL(10,2));
\n""")

    print("Writing Users...")
    f.write("\n".join(to_sql_inserts(df_users, 'users')) + "\n\n")
    print("Writing Products...")
    f.write("\n".join(to_sql_inserts(df_products, 'products')) + "\n\n")
    print("Writing Orders...")
    f.write("\n".join(to_sql_inserts(df_orders, 'orders')) + "\n\n")
    print("Writing Order Items...")
    f.write("\n".join(to_sql_inserts(df_order_items, 'order_items')) + "\n\n")

print("Done! File generated successfully.")

Generating realistic e-commerce data...
Writing Users...
Writing Products...
Writing Orders...
Writing Order Items...
Done! File generated successfully.
